In [0]:
database_name = "fsdh_artemis"

gold_table_mission_metrics = f"{database_name}.gold_orion_mission_metrics"
gold_table_latest_state = f"{database_name}.gold_orion_latest_state"
gold_table_summary = f"{database_name}.gold_orion_summary"
silver_table_metadata = f"{database_name}.silver_oem_metadata"

In [0]:
display(spark.table(gold_table_mission_metrics).orderBy("epoch_utc"))
display(spark.table(gold_table_latest_state))
display(spark.table(gold_table_summary))
display(spark.table(silver_table_metadata).orderBy("line_number"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_speed_over_time AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    speed_km_s
FROM {gold_table_mission_metrics}
ORDER BY epoch_utc
""")

display(spark.table(f"{database_name}.vw_orion_speed_over_time"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_distance_over_time AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    distance_from_origin_km
FROM {gold_table_mission_metrics}
ORDER BY epoch_utc
""")

display(spark.table(f"{database_name}.vw_orion_distance_over_time"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_latest_kpis AS
SELECT
    source_file,
    epoch_utc AS latest_epoch_utc,
    mission_elapsed_hours,
    speed_km_s,
    distance_from_origin_km,
    x_km,
    y_km,
    z_km,
    vx_km_s,
    vy_km_s,
    vz_km_s
FROM {gold_table_latest_state}
""")

display(spark.table(f"{database_name}.vw_orion_latest_kpis"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_summary AS
SELECT
    source_file,
    mission_start_utc,
    mission_end_utc,
    mission_duration_hours,
    state_vector_count,
    min_speed_km_s,
    max_speed_km_s,
    avg_speed_km_s,
    min_distance_from_origin_km,
    max_distance_from_origin_km
FROM {gold_table_summary}
""")

display(spark.table(f"{database_name}.vw_orion_summary"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_position_series AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    x_km,
    y_km,
    z_km
FROM {gold_table_mission_metrics}
ORDER BY epoch_utc
""")

display(spark.table(f"{database_name}.vw_orion_position_series"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_metadata_wide AS
SELECT
    source_file,
    MAX(CASE WHEN meta_key = 'OBJECT_NAME' THEN meta_value END) AS object_name,
    MAX(CASE WHEN meta_key = 'OBJECT_ID' THEN meta_value END) AS object_id,
    MAX(CASE WHEN meta_key = 'CENTER_NAME' THEN meta_value END) AS center_name,
    MAX(CASE WHEN meta_key = 'REF_FRAME' THEN meta_value END) AS ref_frame,
    MAX(CASE WHEN meta_key = 'TIME_SYSTEM' THEN meta_value END) AS time_system,
    MAX(CASE WHEN meta_key = 'START_TIME' THEN meta_value END) AS start_time,
    MAX(CASE WHEN meta_key = 'USEABLE_START_TIME' THEN meta_value END) AS useable_start_time,
    MAX(CASE WHEN meta_key = 'USEABLE_STOP_TIME' THEN meta_value END) AS useable_stop_time,
    MAX(CASE WHEN meta_key = 'STOP_TIME' THEN meta_value END) AS stop_time
FROM {silver_table_metadata}
GROUP BY source_file
""")

display(spark.table(f"{database_name}.vw_orion_metadata_wide"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_exec_overview AS
SELECT
    s.source_file,
    m.object_name,
    m.object_id,
    m.center_name,
    m.ref_frame,
    m.time_system,
    s.mission_start_utc,
    s.mission_end_utc,
    s.mission_duration_hours,
    s.state_vector_count,
    l.latest_epoch_utc,
    l.speed_km_s AS latest_speed_km_s,
    l.distance_from_origin_km AS latest_distance_from_origin_km,
    s.min_speed_km_s,
    s.max_speed_km_s,
    s.avg_speed_km_s,
    s.min_distance_from_origin_km,
    s.max_distance_from_origin_km
FROM {database_name}.vw_orion_summary s
LEFT JOIN {database_name}.vw_orion_latest_kpis l
    ON s.source_file = l.source_file
LEFT JOIN {database_name}.vw_orion_metadata_wide m
    ON s.source_file = m.source_file
""")

display(spark.table(f"{database_name}.vw_orion_exec_overview"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_speed_to_now AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    speed_km_s
FROM {gold_table_mission_metrics}
WHERE epoch_utc <= current_timestamp()
ORDER BY epoch_utc
""")

display(spark.table(f"{database_name}.vw_orion_speed_to_now"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_distance_to_now AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    distance_from_origin_km
FROM {gold_table_mission_metrics}
WHERE epoch_utc <= current_timestamp()
ORDER BY epoch_utc
""")

display(spark.table(f"{database_name}.vw_orion_distance_to_now"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {database_name}.vw_orion_current_state AS
SELECT
    source_file,
    epoch_utc,
    mission_elapsed_hours,
    speed_km_s,
    distance_from_origin_km,
    x_km,
    y_km,
    z_km,
    vx_km_s,
    vy_km_s,
    vz_km_s
FROM {gold_table_latest_state}
""")

display(spark.table(f"{database_name}.vw_orion_current_state"))

In [0]:
for view_name in [
    "vw_orion_speed_over_time",
    "vw_orion_distance_over_time",
    "vw_orion_latest_kpis",
    "vw_orion_summary",
    "vw_orion_position_series",
    "vw_orion_metadata_wide",
    "vw_orion_exec_overview"
]:
    print(f"{database_name}.{view_name}")